# Hunyuan3D-2.1 — 3D Model Generation
Generate 3D models from images using Tencent's Hunyuan3D-2.1.

**Setup**: Runtime > Change runtime type > **T4 GPU** (free) or **A100** (Pro)

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2: Clone and install Hunyuan3D-2.1
!rm -rf /content/Hunyuan3D-2.1
!git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git /content/Hunyuan3D-2.1
%cd /content/Hunyuan3D-2.1

# Install deps — filter out bpy + problematic build-from-source packages
!grep -v -E '^(bpy|basicsr|realesrgan)' requirements.txt | pip install -r /dev/stdin -q
!pip install trimesh -q

# Install basicsr/realesrgan from prebuilt wheels (skip if unavailable)
!pip install basicsr realesrgan -q 2>/dev/null || echo "basicsr/realesrgan skipped (optional for texture upscaling)"

# Build custom_rasterizer (CUDA extension)
!cd hy3dpaint/custom_rasterizer && pip install -e . --no-build-isolation -q

# JIT compile mesh_inpaint_processor (C++ only, no setup.py)
!cd /content/Hunyuan3D-2.1/hy3dpaint/DifferentiableRenderer && python -c "import torch; from torch.utils.cpp_extension import load; load(name='mesh_inpaint_processor', sources=['mesh_inpaint_processor.cpp'], verbose=True)"

# Patch mesh_utils.py — make bpy import optional (trimesh fallback)
import pathlib
mu = pathlib.Path('hy3dpaint/DifferentiableRenderer/mesh_utils.py')
src = mu.read_text()
src = src.replace(
    'import bpy',
    'try:\n    import bpy\n    HAS_BPY = True\nexcept ImportError:\n    HAS_BPY = False'
)
mu.write_text(src)

print('Installation complete!')

In [ ]:
# Cell 3: Upload your input image
from google.colab import files
uploaded = files.upload()
input_image = list(uploaded.keys())[0]
print(f'Uploaded: {input_image}')

In [ ]:
# Cell 4: Stage 1 — Shape Generation
import sys, os
sys.path.insert(0, 'hy3dshape')
sys.path.insert(0, 'hy3dpaint')

from PIL import Image
from hy3dshape import Hunyuan3DDiTFlowMatchingPipeline
from hy3dshape.rembg import BackgroundRemover

print('Loading shape pipeline...')
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2.1', subfolder='hunyuan3d-dit-v2-1')

image = Image.open(input_image)
image = BackgroundRemover()(image)

print('Generating shape...')
mesh = pipeline(
    image=image,
    num_inference_steps=30,
    guidance_scale=7.5,
    octree_resolution=256,
)[0]

os.makedirs('output', exist_ok=True)
mesh.export('output/shape.glb')
print('Shape saved: output/shape.glb')

In [ ]:
# Cell 5: Stage 2 — Texture Generation
from textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig

print('Loading texture pipeline...')
conf = Hunyuan3DPaintConfig(6, 512)
conf.realesrgan_ckpt_path = 'hy3dpaint/ckpt/RealESRGAN_x4plus.pth'
conf.multiview_cfg_path = 'hy3dpaint/cfgs/hunyuan-paint-pbr.yaml'
conf.custom_pipeline = 'hy3dpaint/hunyuanpaintpbr'
paint = Hunyuan3DPaintPipeline(conf)

print('Generating texture...')
paint(
    mesh_path='output/shape.glb',
    image_path=image,
    output_mesh_path='output/textured.obj',
    save_glb=False,
)
print('Textured model saved: output/textured.obj')

In [ ]:
# Cell 6: Convert to GLB and visualize
import trimesh
mesh = trimesh.load('output/textured.obj', process=False)
mesh.export('output/textured.glb', file_type='glb')
print('GLB saved: output/textured.glb')

# Show in notebook
mesh.show()

In [ ]:
# Cell 7: Download results
from google.colab import files
files.download('output/shape.glb')
files.download('output/textured.glb')
files.download('output/textured.obj')
files.download('output/textured.jpg')
print('Downloads started!')